# Notebook 1 — EDA, Feature Engineering & Preprocessing

**Goal:** Understand the data, clean it, create new useful features, and handle the class imbalance problem.

**Run this notebook first.** It saves the processed data so the other notebooks can use it.

In [ ]:
# ── Import all the libraries we need ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('All libraries loaded successfully!')

In [ ]:
# ── Load the dataset ───────────────────────────────────────────────────────────
DATA_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(DATA_PATH)

print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## 1. First Look at the Data

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# ── Check for missing values ──────────────────────────────────────────────────
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

if missing.sum() == 0:
    print('No missing values found!')

In [ ]:
# ── TotalCharges is stored as text — fix that ─────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print('TotalCharges fixed. Missing values now:', df['TotalCharges'].isnull().sum())

## 2. Class Imbalance — The Core Problem

In [ ]:
# ── How many customers churned vs stayed? ────────────────────────────────────
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

print('Churn counts:')
print(churn_counts)
print(f'\nChurn rate: {churn_pct["Yes"]:.1f}%')
print(f'Retained  : {churn_pct["No"]:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#2196F3', '#F44336']
churn_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white', rot=0)
axes[0].set_title('Churn vs Not Churned (Count)', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of Customers')
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=11)

axes[1].pie(churn_counts, labels=['Not Churned', 'Churned'], autopct='%1.1f%%',
            colors=colors, startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Churn Distribution (%)', fontsize=13)

plt.tight_layout()
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/01_class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nThe dataset is imbalanced — we will fix this with SMOTE later.')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Churn rate by contract type ───────────────────────────────────────────────
contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index()
contract_churn.columns = ['Contract', 'ChurnRate']

plt.figure(figsize=(8, 5))
bars = plt.bar(contract_churn['Contract'], contract_churn['ChurnRate'],
               color=['#F44336', '#FF9800', '#4CAF50'], edgecolor='white', width=0.5)
plt.title('Churn Rate by Contract Type', fontsize=13)
plt.ylabel('Churn Rate (%)')
plt.xlabel('Contract Type')
for bar, val in zip(bars, contract_churn['ChurnRate']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontsize=11)
plt.savefig('../reports/figures/02_churn_by_contract.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Churn by tenure and monthly charges ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[df['Churn'] == 'No']['tenure'].plot(kind='hist', bins=30, ax=axes[0],
                                        alpha=0.7, color='#2196F3', label='Not Churned')
df[df['Churn'] == 'Yes']['tenure'].plot(kind='hist', bins=30, ax=axes[0],
                                         alpha=0.7, color='#F44336', label='Churned')
axes[0].set_title('Tenure Distribution by Churn', fontsize=12)
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Count')
axes[0].legend()

df[df['Churn'] == 'No']['MonthlyCharges'].plot(kind='hist', bins=30, ax=axes[1],
                                                 alpha=0.7, color='#2196F3', label='Not Churned')
df[df['Churn'] == 'Yes']['MonthlyCharges'].plot(kind='hist', bins=30, ax=axes[1],
                                                  alpha=0.7, color='#F44336', label='Churned')
axes[1].set_title('Monthly Charges Distribution by Churn', fontsize=12)
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/03_tenure_charges_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Churn rate across multiple categorical features ───────────────────────────
cat_features = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
                'PhoneService', 'InternetService', 'PaymentMethod']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    churn_rate.plot(kind='bar', ax=axes[i], color='#F44336', edgecolor='white', rot=30)
    axes[i].set_title(f'Churn Rate by {col}', fontsize=10)
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].set_xlabel('')

axes[-1].set_visible(False)
plt.suptitle('Churn Rate by Categorical Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/04_categorical_churn_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
corr = df[numeric_cols].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5,
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Between Numeric Features', fontsize=12)
plt.savefig('../reports/figures/05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Note: TotalCharges = tenure x MonthlyCharges roughly, so they are correlated.')

## 4. Feature Engineering

We create 3 new features that the original dataset does not have.

In [ ]:
# ── Define service columns (used here AND in encoding below) ──────────────────
# IMPORTANT: this must be defined before encoding
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

# 1. service_count: how many add-on services the customer has
df['service_count'] = df[service_cols].apply(
    lambda row: sum(row == 'Yes'), axis=1
)

# 2. charge_per_tenure: monthly cost relative to how long they've been a customer
df['charge_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)

# 3. has_premium_services: does the customer have 3 or more add-on services?
df['has_premium_services'] = (df['service_count'] >= 3).astype(int)

print('New features added:')
print(df[['service_count', 'charge_per_tenure', 'has_premium_services']].describe())

In [ ]:
# ── Visualize new features ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

service_churn = df.groupby('service_count')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
)
service_churn.plot(kind='bar', ax=axes[0], color='#9C27B0', edgecolor='white', rot=0)
axes[0].set_title('Churn Rate by Number of Services', fontsize=12)
axes[0].set_xlabel('Number of Services Subscribed')
axes[0].set_ylabel('Churn Rate (%)')

churned     = df[df['Churn'] == 'Yes']['charge_per_tenure']
not_churned = df[df['Churn'] == 'No']['charge_per_tenure']
axes[1].hist(not_churned, bins=40, alpha=0.6, color='#2196F3', label='Not Churned')
axes[1].hist(churned,     bins=40, alpha=0.6, color='#F44336', label='Churned')
axes[1].set_title('Charge per Tenure — Churned vs Not', fontsize=12)
axes[1].set_xlabel('Monthly Charge / Tenure')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/06_engineered_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Encoding — Converting Text to Numbers

Machine learning models only understand numbers. We need to convert all Yes/No and category columns.

In [ ]:
# ── Drop customerID ───────────────────────────────────────────────────────────
df = df.drop(columns=['customerID'])

# ── Convert target column: Churn Yes/No → 1/0 ────────────────────────────────
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# ── Binary Yes/No columns → 1/0 ──────────────────────────────────────────────
# service_cols was defined in Section 4 above
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling'] + service_cols

for col in binary_cols:
    if df[col].dtype == object:
        df[col] = df[col].map(lambda x: 1 if x in ['Yes', 'Female'] else 0)

# ── Multi-category columns → One-Hot Encoding ─────────────────────────────────
multi_cat_cols = ['MultipleLines', 'InternetService', 'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True, dtype=int)

print(f'Final dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

# Verify no text columns remain
text_cols = df.select_dtypes(include='object').columns.tolist()
if text_cols:
    print(f'WARNING: These columns are still text: {text_cols}')
else:
    print('All columns are now numeric. Ready for ML!')

## 6. Train/Test Split

In [ ]:
# ── Separate features (X) from target (y) ────────────────────────────────────
X = df.drop(columns=['Churn'])
y = df['Churn']

# ── Split into 80% training, 20% testing ─────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set  : {X_train.shape[0]} rows')
print(f'Test set      : {X_test.shape[0]} rows')
print(f'Churn in train: {y_train.mean()*100:.1f}%')
print(f'Churn in test : {y_test.mean()*100:.1f}%')

## 7. SMOTE — Fix the Class Imbalance

**The problem:** ~73% non-churners, only ~27% churners.  
**Why it matters:** A model that always says 'no churn' would be 73% accurate but completely useless.  
**SMOTE:** Creates synthetic new examples of churners so the model learns both classes equally.

> **Critical rule:** SMOTE is applied ONLY to training data. Never the test data.

In [ ]:
# ── Scale features before SMOTE ───────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ── Apply SMOTE to training data only ─────────────────────────────────────────
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print('BEFORE SMOTE (training set):')
print(f'  Not Churned: {sum(y_train == 0)}')
print(f'  Churned    : {sum(y_train == 1)}')
print()
print('AFTER SMOTE (training set):')
print(f'  Not Churned: {sum(y_train_balanced == 0)}')
print(f'  Churned    : {sum(y_train_balanced == 1)}')
print()
print('Test set is UNCHANGED (as it should be):')
print(f'  Not Churned: {sum(y_test == 0)}')
print(f'  Churned    : {sum(y_test == 1)}')

In [ ]:
# ── Visualize SMOTE effect ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

before = pd.Series(y_train).value_counts().sort_index()
axes[0].bar(['Not Churned', 'Churned'], before.values, color=['#2196F3', '#F44336'], edgecolor='white')
axes[0].set_title('BEFORE SMOTE', fontsize=12)
axes[0].set_ylabel('Count')
for i, v in enumerate(before.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontsize=11)

after = pd.Series(y_train_balanced).value_counts().sort_index()
axes[1].bar(['Not Churned', 'Churned'], after.values, color=['#2196F3', '#F44336'], edgecolor='white')
axes[1].set_title('AFTER SMOTE', fontsize=12)
axes[1].set_ylabel('Count')
for i, v in enumerate(after.values):
    axes[1].text(i, v + 20, str(v), ha='center', fontsize=11)

plt.suptitle('Effect of SMOTE on Training Data', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/07_smote_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Everything for the Next Notebook

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

pd.DataFrame(X_train_balanced, columns=X.columns).to_csv('../data/processed/X_train.csv', index=False)
pd.DataFrame(y_train_balanced, columns=['Churn']).to_csv('../data/processed/y_train.csv', index=False)

pd.DataFrame(X_test_scaled, columns=X.columns).to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

X.to_csv('../data/processed/X_original.csv', index=False)
y.to_csv('../data/processed/y_original.csv', index=False)

joblib.dump(scaler, '../models/scaler.joblib')
joblib.dump(list(X.columns), '../models/feature_names.joblib')

print('All files saved successfully!')
print('Notebook 1 complete! Run Notebook 2 next.')